####    Hands-on Lab: Working with a real world data-set
#####   LiR
######  0303-2026
######  The dataset is available from the Chicago Data Portal: 
######       https://data.cityofchicago.org/Education/Chicago-Public-Schools-Progress-Report-Cards-2011-/9xs2-f89t
#####   https://labs.cognitiveclass.ai/v2/tools/jupyterlab?ulid=ulid-fb137d799af484f1568dc2543c7f5430d37aea2e


In [3]:
#Connect to the database
#The syntax for connecting to magic sql using sqllite is
#%sql sqlite://db/DatabaseName

In [4]:
import csv, sqlite3

con = sqlite3.connect("DB/RealWorldData.db")
cur = con.cursor()

In [5]:
!pip install pandas
!pip install ipython-sql prettytable

import prettytable
prettytable.DEFAULT = 'DEFAULT'

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [6]:
!pip install ipython-sql
%load_ext sql

Defaulting to user installation because normal site-packages is not writeable


In [7]:
%sql sqlite:///DB/RealWorldData.db

In [12]:
# Store the dataset in a Table
import pandas
df = pandas.read_csv('csv/m5_L1/ChicagoPublicSchools.csv')
# df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv")
df.to_sql("ChicagoPublicSchools_DATA", con, if_exists='replace', index=False,method="multi")
#df

566

In [40]:
#Query the database system catalog to retrieve table metadata
#You can verify that the table creation was successful by retrieving the list of all tables in your schema and checking whether the SCHOOLS table was created
%sql select name from sqlite_master WHERE type='table'

 * sqlite:///DB/RealWorldData.db
Done.


name
ChicagoPublicSchools
ChicagoPublicSchools_DATA


In [15]:
%sql SELECT count(name) FROM PRAGMA_TABLE_INFO('ChicagoPublicSchools_DATA');

 * sqlite:///DB/RealWorldData.db
Done.


count(name)
78


In [18]:
#Now retrieve the the list of columns in SCHOOLS table and their column type (datatype) and length.
%sql SELECT name,type,length(type) FROM PRAGMA_TABLE_INFO('ChicagoPublicSchools_DATA');

 * sqlite:///DB/RealWorldData.db
Done.


name,type,length(type)
School_ID,INTEGER,7
NAME_OF_SCHOOL,TEXT,4
"Elementary, Middle, or High School",TEXT,4
Street_Address,TEXT,4
City,TEXT,4
State,TEXT,4
ZIP_Code,INTEGER,7
Phone_Number,TEXT,4
Link,TEXT,4
Network_Manager,TEXT,4


In [ ]:
# Problem 
# How many Elementary Schools are in the dataset?



In [49]:
#%sql SELECT [Elementary, Middle, or High School] FROM ChicagoPublicSchools_DATA;

In [52]:
#df or
#%sql select * from ChicagoPublicSchools_DATA
%sql select count(*) from  ChicagoPublicSchools_DATA where TRIM([Elementary, Middle, or High School])='ES';
#%sql select count(*) 
#from ChicagoPublicSchools_DATA 
#where TRIM([Elementary, Middle, or High School"]) = 'ES';


 * sqlite:///DB/RealWorldData.db
Done.


count(*)
462


In [65]:
#Problem 2 What is the highest Safety Score?
#df
%sql select MAX([SAFETY_SCORE]) AS MAX_Safety_Score from  ChicagoPublicSchools_DATA;


 * sqlite:///DB/RealWorldData.db
Done.


MAX_Safety_Score
99.0


In [73]:
#Problem 3 Which schools have highest Safety Score?
%sql select count(*) as Count_School  from  ChicagoPublicSchools_DATA where [SAFETY_SCORE] = (select MAX([SAFETY_SCORE]) AS MAX_Safety_Score from  ChicagoPublicSchools_DATA);

 * sqlite:///DB/RealWorldData.db
Done.


Count_School
19


In [92]:
#Problem 4 What are the top 10 schools with the highest "Average Student Attendance"?
#%sql select *  from  ChicagoPublicSchools_DATA LIMIT 10 where [AVERAGE_STUDENT_ATTENDANC]=(select max([AVERAGE_STUDENT_ATTENDANCE])  from  ChicagoPublicSchools_DATA) ;
%sql    SELECT  NAME_OF_SCHOOL, AVERAGE_STUDENT_ATTENDANCE \
        FROM ChicagoPublicSchools_DATA \
        ORDER BY AVERAGE_STUDENT_ATTENDANCE DESC nulls last limit 10 ;

 * sqlite:///DB/RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
John Charles Haines Elementary School,98.40%
James Ward Elementary School,97.80%
Edgar Allan Poe Elementary Classical School,97.60%
Orozco Fine Arts & Sciences Elementary School,97.60%
Rachel Carson Elementary School,97.60%
Annie Keller Elementary Gifted Magnet School,97.50%
Andrew Jackson Elementary Language Academy,97.40%
Lenart Elementary Regional Gifted Center,97.40%
Disney II Magnet School,97.30%
John H Vanderpoel Elementary Magnet School,97.20%


In [93]:
#Problem 5 Retrieve the list of 5 Schools with the lowest Average Student Attendance sorted in ascending order based on attendance
%sql    SELECT  NAME_OF_SCHOOL, AVERAGE_STUDENT_ATTENDANCE \
        FROM ChicagoPublicSchools_DATA \
        ORDER BY AVERAGE_STUDENT_ATTENDANCE ASC nulls last limit 5 ;

 * sqlite:///DB/RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
Richard T Crane Technical Preparatory High School,57.90%
Barbara Vick Early Childhood & Family Center,60.90%
Dyett High School,62.50%
Wendell Phillips Academy High School,63.00%
Orr Academy High School,66.30%


In [95]:
#Problem 6 Now remove the '%' sign from the above result set for Average Student Attendance column
%sql    SELECT NAME_OF_SCHOOL,REPLACE(AVERAGE_STUDENT_ATTENDANCE, '%', '') AS Clean_Attendance \
        FROM ChicagoPublicSchools_DATA \
        ORDER BY CAST(REPLACE(AVERAGE_STUDENT_ATTENDANCE, '%', '') AS REAL) ASC \
        LIMIT 5;

 * sqlite:///DB/RealWorldData.db
Done.


NAME_OF_SCHOOL,Clean_Attendance
Velma F Thomas Early Childhood Center,None
Richard T Crane Technical Preparatory High School,57.90
Barbara Vick Early Childhood & Family Center,60.90
Dyett High School,62.50
Wendell Phillips Academy High School,63.00


In [98]:
#Problem 7 Which Schools have Average Student Attendance lower than 70%?
#%sql    SELECT  NAME_OF_SCHOOL, AVERAGE_STUDENT_ATTENDANCE \
#        FROM ChicagoPublicSchools_DATA \
#        where  AVERAGE_STUDENT_ATTENDANCE < '70%';

%sql SELECT Name_of_School, Average_Student_Attendance  \
     from ChicagoPublicSchools_DATA \
     where CAST ( REPLACE(Average_Student_Attendance, '%', '') AS DOUBLE ) < 70 \
     order by Average_Student_Attendance


 * sqlite:///DB/RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
Richard T Crane Technical Preparatory High School,57.90%
Barbara Vick Early Childhood & Family Center,60.90%
Dyett High School,62.50%
Wendell Phillips Academy High School,63.00%
Orr Academy High School,66.30%
Manley Career Academy High School,66.80%
Chicago Vocational Career Academy High School,68.80%
Roberto Clemente Community Academy High School,69.60%


In [114]:
#Problem 8 Get the total College Enrollment for each Community Area
%sql    SELECT  COMMUNITY_AREA_NAME, SUM(COLLEGE_ENROLLMENT) AS TOTAL_ENROLLMENT \
        FROM ChicagoPublicSchools_DATA \
        GROUP BY [COMMUNITY_AREA_NAME] \
        order by TOTAL_ENROLLMENT;
#College_Eligibility__, College_Enrollment_Rate_, COMMUNITY_AREA_NUMBER, COMMUNITY_AREA_NAME

 * sqlite:///DB/RealWorldData.db
Done.


COMMUNITY_AREA_NAME,TOTAL_ENROLLMENT
OAKLAND,140
FULLER PARK,531
BURNSIDE,549
OHARE,786
LOOP,871
EDISON PARK,910
HEGEWISCH,963
MONTCLARE,1317
NEAR SOUTH SIDE,1378
FOREST GLEN,1431


In [118]:
#Problem 9 Get the 5 Community Areas with the least total College Enrollment sorted in ascending order
%sql    SELECT COMMUNITY_AREA_NAME,SUM(COLLEGE_ENROLLMENT) AS TOTAL_ENROLLMENT \
        FROM ChicagoPublicSchools_DATA \
        GROUP BY COMMUNITY_AREA_NAME \
        ORDER BY TOTAL_ENROLLMENT ASC \
        LIMIT 5;

 * sqlite:///DB/RealWorldData.db
Done.


COMMUNITY_AREA_NAME,TOTAL_ENROLLMENT
OAKLAND,140
FULLER PARK,531
BURNSIDE,549
OHARE,786
LOOP,871


In [126]:
#Problem 10 List 5 schools with lowest safety score.
#вернет все с мин значением
#%sql    SELECT NAME_OF_SCHOOL, SAFETY_SCORE,COMMUNITY_AREA_NAME, * \
#        FROM ChicagoPublicSchools_DATA \
#        where SAFETY_SCORE = (select min(SAFETY_SCORE) AS MAX_Safety_Score from  ChicagoPublicSchools_DATA) \
#        LIMIT 5;
#-------------вернет тольк пять с низким значением
%sql    SELECT NAME_OF_SCHOOL,SAFETY_SCORE, COMMUNITY_AREA_NAME \
        FROM ChicagoPublicSchools_DATA \
        where SAFETY_SCORE IS NOT NULL AND SAFETY_SCORE > 0 \
        ORDER BY SAFETY_SCORE ASC \
        LIMIT 5;

 * sqlite:///DB/RealWorldData.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE,COMMUNITY_AREA_NAME
Edmond Burke Elementary School,1.0,WASHINGTON PARK
Luke O'Toole Elementary School,5.0,WEST ENGLEWOOD
George W Tilton Elementary School,6.0,WEST GARFIELD PARK
Foster Park Elementary School,11.0,AUBURN GRESHAM
Emil G Hirsch Metropolitan High School,13.0,GREATER GRAND CROSSING


In [143]:
#Problem 11 Get the hardship index for the community area of the school which has College Enrollment of 4368
%sql    SELECT *  \
        FROM ChicagoPublicSchools_DATA \
        where COLLEGE_ENROLLMENT = 4368;
# нет тут поля hardship_index нужно брать из 
# df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv")

 * sqlite:///DB/RealWorldData.db
Done.


School_ID,NAME_OF_SCHOOL,"Elementary, Middle, or High School",Street_Address,City,State,ZIP_Code,Phone_Number,Link,Network_Manager,Collaborative_Name,Adequate_Yearly_Progress_Made_,Track_Schedule,CPS_Performance_Policy_Status,CPS_Performance_Policy_Level,HEALTHY_SCHOOL_CERTIFIED,Safety_Icon,SAFETY_SCORE,Family_Involvement_Icon,Family_Involvement_Score,Environment_Icon,Environment_Score,Instruction_Icon,Instruction_Score,Leaders_Icon,Leaders_Score,Teachers_Icon,Teachers_Score,Parent_Engagement_Icon,Parent_Engagement_Score,Parent_Environment_Icon,Parent_Environment_Score,AVERAGE_STUDENT_ATTENDANCE,Rate_of_Misconducts__per_100_students_,Average_Teacher_Attendance,Individualized_Education_Program_Compliance_Rate,Pk_2_Literacy__,Pk_2_Math__,Gr3_5_Grade_Level_Math__,Gr3_5_Grade_Level_Read__,Gr3_5_Keep_Pace_Read__,Gr3_5_Keep_Pace_Math__,Gr6_8_Grade_Level_Math__,Gr6_8_Grade_Level_Read__,Gr6_8_Keep_Pace_Math_,Gr6_8_Keep_Pace_Read__,Gr_8_Explore_Math__,Gr_8_Explore_Read__,ISAT_Exceeding_Math__,ISAT_Exceeding_Reading__,ISAT_Value_Add_Math,ISAT_Value_Add_Read,ISAT_Value_Add_Color_Math,ISAT_Value_Add_Color_Read,Students_Taking__Algebra__,Students_Passing__Algebra__,9th Grade EXPLORE (2009),9th Grade EXPLORE (2010),10th Grade PLAN (2009),10th Grade PLAN (2010),Net_Change_EXPLORE_and_PLAN,11th Grade Average ACT (2011),Net_Change_PLAN_and_ACT,College_Eligibility__,Graduation_Rate__,College_Enrollment_Rate__,COLLEGE_ENROLLMENT,General_Services_Route,Freshman_on_Track_Rate__,X_COORDINATE,Y_COORDINATE,Latitude,Longitude,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,Ward,Police_District,Location
609720,Albert G Lane Technical High School,HS,2501 W Addison St,Chicago,IL,60618,(773) 534-5400,http://schoolreports.cps.edu/SchoolProgressReport_Eng/Spring2011Eng_609720.pdf,North-Northwest Side High School Network,NORTH-NORTHWEST SIDE COLLABORATIVE,Yes,Standard,Not on Probation,Level 1,No,Very Strong,88.0,NDA,NDA,Strong,62.0,Average,52.0,Weak,NDA,NDA,NDA,NDA,NDA,NDA,NDA,96.30%,2.1,96.20%,99.40%,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,None,None,None,None,NDA,NDA,NDA,NDA,19.1,19.5,19.9,20.1,1,23.4,3.5,67.9,92.2,79.8,4368,35,90.7,1158975.392,1923791.705,41.94661693,-87.69105603,5,NORTH CENTER,47,19,"(41.94661693, -87.69105603)"


In [145]:
#Problem11
df2 = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv")
df2.to_sql("ChicagoCensus_DATA", con, if_exists='replace', index=False,method="multi")
df2

,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0
...,...,...,...,...,...,...,...,...,...
73,74.0,Mount Greenwood,1.0,3.4,8.7,4.3,36.8,34381,16.0
74,75.0,Morgan Park,0.8,13.2,15.0,10.8,40.3,27149,30.0
75,76.0,O'Hare,3.6,15.4,7.1,10.9,30.3,25828,24.0
76,77.0,Edgewater,4.1,18.2,9.2,9.7,23.8,33385,19.0


In [152]:
%%sql 
    select hardship_index 
    from ChicagoCensus_DATA CD, ChicagoPublicSchools_DATA CPS 
    where CD.community_area_number = CPS.community_area_number 
    and college_enrollment = 4368

 * sqlite:///DB/RealWorldData.db
Done.


HARDSHIP_INDEX
6.0


In [175]:
#Problem 12 Get the hardship index for the community area which has the highest value for College Enrollment
#%%sql  через Magic
#%%sql  via Python
%sql \
select hardship_index \
from ChicagoCensus_DATA CD, ChicagoPublicSchools_DATA CPS \
where CD.community_area_number = CPS.community_area_number \
and CPS.COLLEGE_ENROLLMENT = (SELECT max(COLLEGE_ENROLLMENT) FROM ChicagoPublicSchools_DATA);

 * sqlite:///DB/RealWorldData.db
Done.


HARDSHIP_INDEX
6.0


In [171]:
%%sql
SELECT CD.HARDSHIP_INDEX
FROM ChicagoCensus_DATA CD
JOIN ChicagoPublicSchools_DATA CPS
  ON CD.community_area_number = CPS.community_area_number
WHERE CPS.COLLEGE_ENROLLMENT = (
    SELECT MAX(COLLEGE_ENROLLMENT)
    FROM ChicagoPublicSchools_DATA
);

 * sqlite:///DB/RealWorldData.db
Done.


HARDSHIP_INDEX
6.0


In [159]:
%sql SELECT max(COLLEGE_ENROLLMENT) AS max_TOTAL_ENROLLMENT  FROM ChicagoPublicSchools_DATA

 * sqlite:///DB/RealWorldData.db
Done.


max_TOTAL_ENROLLMENT
4368
